In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Vacuum Notebook

# COMMAND ----------

# MAGIC %md
# MAGIC ### Imports

# COMMAND ----------

from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import datetime
import re

# COMMAND ----------

# MAGIC %md
# MAGIC **Catalog Selection**

# COMMAND ----------

# Fetch catalog parameter
catalog_param = dbutils.widgets.get("catalog").strip()
audit_catalog = dbutils.widgets.get("auditcatalog")
audit_schema = dbutils.widgets.get("auditschema")
excl_catalog_param = dbutils.widgets.get("exclude_catalog").strip()

# If no catalog is provided, fetch all available catalogs
if not catalog_param:
    print("No catalog provided. Fetching all available catalogs...")
    all_catalogs_df = spark.sql("SHOW CATALOGS")
    catalog_to_vacuum = [row.catalog for row in all_catalogs_df.collect() if row.catalog.lower() not in {excl_catalog_param}]
else:
    catalog_to_vacuum = [x.strip() for x in catalog_param.split(",")]

print(f"Catalogs to process: {catalog_to_vacuum}")

# COMMAND ----------

# MAGIC %md
# MAGIC **Default Values**

# COMMAND ----------

default_retention = 168  
#audit_catalog = "databrick"
#audit_schema = "audit"
MAX_WORKERS = 15
run_timestamp = datetime.datetime.now() 

# COMMAND ----------

# MAGIC %md
# MAGIC ### Functions to fetch table properties

# COMMAND ----------


def get_table_details(catalog: str, schema: str, table: str, default_retention: int = 168) -> dict:
    """Get table details including location, size, and file count"""
    try:
        # Fetch table details using DESCRIBE DETAIL
        details = spark.sql(f"DESCRIBE DETAIL `{catalog}`.`{schema}`.`{table}`").collect()[0]
        
        # Extract location
        location = details.location
        
        # Extract retention_hours if the property exists, else use a default value
        retention_hours = None
        
        # Check for the 'delta.deletedFileRetentionDuration' property in the properties dictionary
        if "delta.deletedFileRetentionDuration" in details.properties:
            retention_str = details.properties["delta.deletedFileRetentionDuration"]
            retention_match = re.search(r"(\d+)\s+days?", retention_str)
            if retention_match:
                retention_hours = int(retention_match.group(1)) * 24  # Convert days to hours
            else:
                print(f"Unexpected format for deletedFileRetentionDuration: {retention_str}")
        
        # Set a default retention value if not specified
        if retention_hours is None:
            retention_hours = default_retention

        return {
            "location": location,
            "retention_hours": retention_hours,
        }
    except Exception as e:
        print(f"Error getting details for {catalog}.{schema}.{table}: {str(e)}")
        return None


# COMMAND ----------

# MAGIC %md
# MAGIC ## Function to perform vaccum

# COMMAND ----------

def vacuum_table(catalog: str, schema: str, table: str, retention_hours: int) -> dict:
    try:
        """Vacuum table and return vacuum metrics from both start and end operations"""
        # Execute vacuum command
        print(f"VACUUM {catalog}.{schema}.{table} RETAIN {retention_hours} HOURS")
        spark.sql(f"VACUUM {catalog}.{schema}.{table} RETAIN {retention_hours} HOURS")
        
        # Get the latest two versions of history to capture vacuum metrics
        history_rows = spark.sql(
            f"""
            SELECT *
            FROM (
                SELECT *,
                    row_number() OVER (ORDER BY version DESC) as rn
                FROM (DESCRIBE HISTORY {catalog}.{schema}.{table})
            ) ranked
            WHERE rn <= 2
            ORDER BY version DESC
            """
        ).collect()
        
        # Find vacuum start and end operations
        vacuum_end = next((row for row in history_rows if row.operation == 'VACUUM END'), None)
        vacuum_start = next((row for row in history_rows if row.operation == 'VACUUM START'), None)
        
        if vacuum_start and vacuum_end:
            return {
                "status": "Success",
                'numFilesToDelete': vacuum_start.operationMetrics.get('numFilesToDelete'),
                'sizeOfDataToDelete': vacuum_start.operationMetrics.get('sizeOfDataToDelete'),
                'numDeletedFiles': vacuum_end.operationMetrics.get('numDeletedFiles'),
                'numVacuumedDirectories': vacuum_end.operationMetrics.get('numVacuumedDirectories')
                }
        
        else:
            return {
                "status": "Failed",
                'numFilesToDelete':None,
                'sizeOfDataToDelete': None,
                'numDeletedFiles': None,
                'numVacuumedDirectories': None
            }
    except Exception as e:
        print(f"Error vacuuming {catalog}.{schema}.{table}: {str(e)}")
        return {
                "status": "Failed",
                'numFilesToDelete':None,
                'sizeOfDataToDelete': None,
                'numDeletedFiles': None,
                'numVacuumedDirectories': None
            }

# COMMAND ----------

# MAGIC %md
# MAGIC ##Parallel Vacuum Execution

# COMMAND ----------

def vacuum_table_parallel(catalog: str, schema: str, table: str):
    """Wrapper function for parallel execution."""
    details = get_table_details(catalog, schema, table)
    if details:
        return {
            "run_id": runId, "job_id": jobId, "job_name": job_name, "workspace_id": workspace_id, 
            "user_name": user, "catalog_name": catalog, "schema_name": schema, "table_name": table,
            "location": details["location"], "retention_hours": details["retention_hours"],
            **vacuum_table(catalog, schema, table, details["retention_hours"]),
            "timestamp":run_timestamp
        }
    return None

# COMMAND ----------

# MAGIC %md
# MAGIC # Create audit Table

# COMMAND ----------

# MAGIC %md
# MAGIC **Job Metadata**

# COMMAND ----------

jobId = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("jobId").get()
runId = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("runId").get()
job_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("jobName").get()
user = spark.sql("select current_user()").collect()[0][0]
workspaceUrl = spark.conf.get('spark.databricks.workspaceUrl')
workspace_id_match = re.search(r"adb-(\d+)", workspaceUrl)
workspace_id = workspace_id_match.group(1) if workspace_id_match else "Unknown"

# COMMAND ----------


def create_audit_table():
    """Create the audit table if it doesn't exist"""
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {audit_catalog}.{audit_schema}.vacuum_log (
        run_id STRING,
        job_id STRING,
        job_name STRING,
        workspace_id STRING,
        user_name STRING,
        catalog_name STRING,
        schema_name STRING,
        table_name STRING,
        location STRING,
        retention_hours BIGINT,
        size_of_data_to_delete BIGINT,
        num_files_to_delete BIGINT,
        num_deleted_files BIGINT,
        num_vacuumed_directories BIGINT,
        status STRING,
        timestamp TIMESTAMP
    )
    """
    spark.sql(create_table_sql)
    print(f"Audit table created/verified in {audit_catalog}.{audit_schema}")


# Ensure audit table exists
create_audit_table()
    

# COMMAND ----------

# MAGIC %md
# MAGIC ### Vacuum Process

# COMMAND ----------

vacuum_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_table = {}
    
    for catalog_name in catalog_to_vacuum:
        schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")
        for schema in schemas_df.collect():
            schema_name = schema.databaseName
            if schema_name.lower() == "information_schema":
                print(f"Skipping information_schema: {catalog_name}.{schema_name}")
                continue
            print(f"Processing schema: {catalog_name}.{schema_name}")

            tables_metadata_df = spark.sql(f"""
                SELECT table_name, table_type 
                FROM `{catalog_name}`.information_schema.tables 
                WHERE table_schema = '{schema_name}'
            """)
            
            for row in tables_metadata_df.collect():
                table_name = row.table_name
                table_type = row.table_type
                if table_type.lower() == "view":
                    print(f"Skipping view: {catalog_name}.{schema_name}.{table_name}")
                    continue
                print(f"Processing table: {catalog_name}.{schema_name}.{table_name}")

                future = executor.submit(vacuum_table_parallel, catalog_name, schema_name, table_name)
                future_to_table[future] = (catalog_name, schema_name, table_name)

    for future in as_completed(future_to_table):
        result = future.result()
        if result:
            vacuum_results.append(result)

# COMMAND ----------

# MAGIC %md
# MAGIC **Create** **Audit** **Dataframe**

# COMMAND ----------

audit_df = spark.createDataFrame(vacuum_results)
audit_df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC **Handling Numerical Columns**

# COMMAND ----------

audit_df = audit_df.withColumn("size_of_data_to_delete", F.col("sizeOfDataToDelete").cast(LongType())) \
                   .withColumn("num_files_to_delete", F.col("numFilesToDelete").cast(LongType())) \
                   .withColumn("num_deleted_files", F.col("numDeletedFiles").cast(LongType())) \
                   .withColumn("num_vacuumed_directories", F.col("numVacuumedDirectories").cast(LongType()))

# COMMAND ----------

# MAGIC %md
# MAGIC **Drop unused column and schema verify after transformation**

# COMMAND ----------

audit_df = audit_df.drop("sizeOfDataToDelete", "numFilesToDelete", "numDeletedFiles", "numVacuumedDirectories")
audit_df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ### Write Audit Results

# COMMAND ----------

audit_df.write.mode("append").saveAsTable(f"{audit_catalog}.{audit_schema}.vacuum_log")
print("Audit results saved.")